In [6]:
from qdrant_client import QdrantClient

client = QdrantClient(url="http://localhost:6333")

In [19]:
client.delete_collection(collection_name="document_collection")


True

In [7]:
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from qdrant_client import QdrantClient, models 
from langchain_openai import OpenAIEmbeddings

# def retriever_init(client, collection_name, sparse_embeddings, dense_embeddings) -> QdrantVectorStore:
#     """Initialize a QdrantVectorStore backed by the provided client.

#     If the collection already exists we wrap it directly; otherwise we create a new
#     collection with dense and sparse vector configurations. Explicit ``vector_name``
#     and ``sparse_vector_name`` arguments are provided to avoid validation errors when
#     re‑using an existing collection.
#     """
#     qdrant = None

#     # When the collection already exists we can simply wrap the existing ``QdrantClient``
#     # instance with a ``QdrantVectorStore``. The ``from_existing_collection`` helper
#     # expects connection parameters (url, location, …) and does **not** accept a ``client``
#     # keyword argument, which caused a ``TypeError``. Instead we instantiate the store
#     # directly using the class constructor which does accept a ``client``.
#     if client.collection_exists(collection_name=collection_name):
#         # The existing collection is expected to have a dense vector named "dense"
#         # and a sparse vector named "sparse" (as created below). Explicitly
#         # provide these names so that QdrantVectorStore validates correctly.

#         qdrant = QdrantVectorStore.from_existing_collection(
#             collection_name=collection_name,
#             embedding=dense_embeddings,
#             sparse_embedding=sparse_embeddings,
#             retrieval_mode=RetrievalMode.HYBRID,
#             vector_name="dense",
#             sparse_vector_name="sparse"        )

#     else:
#         # Determine the correct size for dense vectors.
#         # ``OpenAIEmbeddings`` (and many other LangChain embeddings) may not expose a
#         # ``dimensions`` attribute, resulting in ``None`` and causing a ``pydantic``
#         # validation error when creating the Qdrant collection.
#         # If ``dimensions`` is missing, we fall back to a sensible default that
#         # matches the default OpenAI embedding model (text‑embedding‑3‑large → 3072).
#         if getattr(dense_embeddings, "dimensions", None) is None:
#             _dense_dim: int = 3072
#         else:
#             _dense_dim = int(dense_embeddings.dimensions)

#         client.create_collection(
#             collection_name=collection_name,
#             vectors_config={
#                 "dense": models.VectorParams(size=_dense_dim, distance=models.Distance.COSINE)
#             },
#             sparse_vectors_config={"sparse": models.SparseVectorParams()},
#         )

#         # After creating the collection we instantiate the store directly,
#         # specifying the vector names to match the collection configuration.
#         qdrant = QdrantVectorStore(
#             client=client,
#             collection_name=collection_name,
#             embedding=dense_embeddings,
#             sparse_embedding=sparse_embeddings,
#             retrieval_mode=RetrievalMode.HYBRID,
#             vector_name="dense",
#             sparse_vector_name="sparse",
#         )

#     return qdrant
# sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")


# dense_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
# qdrant = retriever_init(
#     client=client,
#     collection_name="document_collection",
#     sparse_embeddings=sparse_embeddings,
#     dense_embeddings=dense_embeddings
# )
# qdrant = QdrantVectorStore.from_existing_collection(
#     collection_name="document_collection",
#     embedding=dense_embeddings,
#     sparse_embedding=sparse_embeddings,
#     retrieval_mode = RetrievalMode.HYBRID,
#     vector_name = "dense",
#     sparse_vector_name = "sparse"
#     )

In [11]:
from langchain_core.tools import create_retriever_tool 
def retriever_init(client, collection_name, sparse_embeddings, dense_embeddings) -> QdrantVectorStore:
    """Initialize a QdrantVectorStore backed by the provided client.

    If the collection already exists we wrap it directly; otherwise we create a new
    collection with dense and sparse vector configurations. Explicit ``vector_name``
    and ``sparse_vector_name`` arguments are provided to avoid validation errors when
    re‑using an existing collection.
    """
    qdrant = None

    # When the collection already exists we can simply wrap the existing ``QdrantClient``
    # instance with a ``QdrantVectorStore``. The ``from_existing_collection`` helper
    # expects connection parameters (url, location, …) and does **not** accept a ``client``
    # keyword argument, which caused a ``TypeError``. Instead we instantiate the store
    # directly using the class constructor which does accept a ``client``.
    if client.collection_exists(collection_name=collection_name):
        # The existing collection is expected to have a dense vector named "dense"
        # and a sparse vector named "sparse" (as created below). Explicitly
        # provide these names so that QdrantVectorStore validates correctly.

        qdrant = QdrantVectorStore.from_existing_collection(
            collection_name=collection_name,
            embedding=dense_embeddings,
            sparse_embedding=sparse_embeddings,
            retrieval_mode=RetrievalMode.HYBRID,
            vector_name="dense",
            sparse_vector_name="sparse",
            validate_collection_config=False
        )

    else:
        # Determine the correct size for dense vectors.
        # ``OpenAIEmbeddings`` (and many other LangChain embeddings) may not expose a
        # ``dimensions`` attribute, resulting in ``None`` and causing a ``pydantic``
        # validation error when creating the Qdrant collection.
        # If ``dimensions`` is missing, we fall back to a sensible default that
        # matches the default OpenAI embedding model (text‑embedding‑3‑large → 3072).
        if getattr(dense_embeddings, "dimensions", None) is None:
            _dense_dim: int = 3072
        else:
            _dense_dim = int(dense_embeddings.dimensions)

        client.create_collection(
            collection_name=collection_name,
            vectors_config={
                "dense": models.VectorParams(size=_dense_dim, distance=models.Distance.COSINE)
            },
            sparse_vectors_config={"sparse": models.SparseVectorParams()},
        )

        # After creating the collection we instantiate the store directly,
        # specifying the vector names to match the collection configuration.
        qdrant = QdrantVectorStore(
            client=client,
            collection_name=collection_name,
            embedding=dense_embeddings,
            sparse_embedding=sparse_embeddings,
            retrieval_mode=RetrievalMode.HYBRID,
            vector_name="dense",
            sparse_vector_name="sparse",
        )

    retriever = qdrant.as_retriever(search_kwargs={"k": 1})

    return create_retriever_tool(
        retriever,
        "document_retriever",
        description="Search for relevant documents to answer user questions",
    )

In [12]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import create_retriever_tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from dotenv import load_dotenv
from dataclasses import dataclass


SYSTEM_PROMPT = """You are a document Q&A assistant.
Answer ONLY using the context provided below.
For each claim, cite the chunk ID in [brackets].
If the context does not contain the answer, say:
'I cannot find this in the provided documents.'
Never fabricate information."""

model = init_chat_model(model="gpt-5.5", temperature=0.1)

checkpointer = InMemorySaver()

config = {'configurable' : {'thread_id': '1'}}

@dataclass
class ResponseFormat():
    summary: str
    chunk_id: str
    page_number: int
    source: str

retriever_tool = retriever_init(
    client=client,
    collection_name="document_collection",
    sparse_embeddings=sparse_embeddings,
    dense_embeddings=dense_embeddings,
)


agent = create_agent(
    model=model,
    tools=[retriever_tool],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer
)




In [14]:
query = "What is Home Credit?"

In [15]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": query},
    ]
    },
    config=config
)

In [16]:
result

{'messages': [HumanMessage(content='How long did he work at Home Credit?', additional_kwargs={}, response_metadata={}, id='e9daff1c-9e78-4e3f-8692-5be0e3f3e3f5'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-5.5-2026-04-23', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--019fec83-67d7-7fa2-9502-45ba9898410c', tool_calls=[{'name': 'document_retriever', 'args': {'query': '"Home Credit" "worked"'}, 'id': 'call_blwGWVuPSfUIQNUctEJD9RiE', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 203, 'output_tokens': 46, 'total_tokens': 249, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 15}}),
  ToolMessage(content='Khoa Bui\n\x83 (437) 443-7831\n# bndk2108@gmail.com\nï linkedin.com/in/bndk2108\n§ github.com/hyolee1999\nEducation\nHo Chi Minh University of Technology\nOct 2017 – April 2022\nBachelor of Computer Science\nHo C

In [59]:
query

'What is Home Credit?'

In [18]:
for chunk in agent.stream(
    {
        'messages':[
            {
                'role':'user',
                'content': 'How long did he work at Home Credit?'
            }
        ]
    },
    config = config,
    stream_mode = "messages",
    version = "v2"
):
    # print(chunk)
    
    if chunk["type"] == "messages":
        token, metadata = chunk["data"]
        # print(token)
        # print(token.content_blocks)
        if token.content_blocks:
            if "text" in token.content_blocks[0]:
                print(token.content_blocks[0]["text"])
            # print(token.content_blocks)
            # print(token.content_blocks[0].text)


He
 worked
 at
 Home
 Credit
 for
 **
3
 years
**,
 from
 **
Oct
 
202
2
 to
 Oct
 
202
5
**
.
 [
K
hoa
 B
ui
]
